# E2-05 · Correlación y regresión lineal

**Estadística II · Ingeniería Industrial · Universidad Industrial de Santander**  
*Versión docente (con soluciones)*

**Propósito de aprendizaje**

- Estimar e interpretar una recta de regresión simple, su R² y la prueba sobre la pendiente.
- Construir intervalos de confianza para la media e intervalos de predicción para una observación nueva.
- Ajustar una regresión múltiple con variables categóricas, validar supuestos con los residuos y comparar modelos.
- Reconocer cuándo **no** hay relación y evitar sobreinterpretar un modelo.

**Duración sugerida:** 2 sesiones de 2 horas  
**Datos:** casos IC08 (quiebres de inventario), IC10 (consumo de combustible) e IC06 (condiciones ergonómicas)

## Estructura

| Parte | Caso | Modalidad | Pregunta |
|---|---|---|---|
| 1 | IC08 · Quiebres de inventario | guiada | ¿La variabilidad de la demanda explica los quiebres? |
| 2 | IC10 · Consumo de combustible | semi-guiada | ¿Qué características del viaje explican el consumo de la flota? |
| 3 | IC06 · Condiciones ergonómicas | autónoma | ¿La temperatura, el ruido o la postura estática se relacionan con la frecuencia cardiaca? |

**Modelo de regresión lineal simple**

$$Y_i = \beta_0 + \beta_1 X_i + \varepsilon_i, \qquad \varepsilon_i \sim N(0, \sigma^2) \text{ independientes.}$$

Los supuestos (linealidad, independencia, varianza constante y normalidad de los errores) se verifican con los **residuos**.

> **📌 Nota para el docente**
>
> **Resultados esperados con las bases del repositorio:**
> - **IC08:** r ≈ 0,64 entre quiebres y variabilidad de la demanda; pendiente ≈ 0,21 puntos de quiebre por cada punto de CV; R² ≈ 0,40. El costo unitario no aporta (p ≈ 0,11). Los residuos son compatibles con normalidad (Shapiro p ≈ 0,44), pero hay registros con 0 % de quiebres (la variable está acotada en 0) y el intervalo de predicción para CV bajos llega a valores negativos: buen punto de discusión.
> - **IC10:** el consumo depende sobre todo del tipo de vehículo y del perfil de ruta; la carga es significativa (≈ 0,44 L/100 km por tonelada) pero está muy correlacionada con el tipo de vehículo (VIF ≈ 6–9), por lo que su coeficiente simple (r ≈ 0,85) se reduce mucho en el modelo múltiple. El día de la semana y la experiencia del conductor no son significativos. R² ajustado ≈ 0,91.
> - **IC06:** todas las correlaciones con la frecuencia cardiaca son prácticamente cero (|r| < 0,06) y el R² del modelo múltiple es ≈ 0,006. Es un caso de **ausencia de relación**: el aprendizaje está en no forzar conclusiones.

# 0. Preparación

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
sns.set_theme(style="whitegrid")
AZUL, CORAL = "#00ADC6", "#FF6F49"

import os
from pathlib import Path

def cargar_csv(nombre, carpeta="casos_industriales/csv"):
    """Busca el archivo en la carpeta actual o en la carpeta datos/ del repositorio.
    Si no lo encuentra y se está en Google Colab, abre el diálogo para subirlo."""
    candidatos = [Path(nombre)] + [Path(p) / "datos" / carpeta / nombre for p in [".", "..", "../..", "../../.."]]
    for ruta in candidatos:
        if ruta.exists():
            print(f"Leyendo {ruta}")
            return pd.read_csv(ruta)
    try:
        from google.colab import files  # solo existe en Colab
        print(f"No encontré '{nombre}'. Súbalo desde su computador:")
        subido = files.upload()
        return pd.read_csv(next(iter(subido)))
    except ImportError:
        raise FileNotFoundError(f"No encontré '{nombre}'. Cópielo en la misma carpeta de este notebook.")


def graficos_residuos(modelo, titulo=""):
    """Residuos vs. ajustados, Q-Q e histograma de residuos."""
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].scatter(modelo.fittedvalues, modelo.resid, s=12, color=AZUL, alpha=.6)
    ax[0].axhline(0, color=CORAL, ls="--"); ax[0].set_xlabel("valores ajustados"); ax[0].set_ylabel("residuos")
    ax[0].set_title("Residuos vs. ajustados")
    sm.qqplot(modelo.resid, line="s", ax=ax[1], markerfacecolor=AZUL, markeredgecolor=AZUL, alpha=.6); ax[1].set_title("Q-Q de residuos")
    ax[2].hist(modelo.resid, bins=25, color=AZUL); ax[2].set_title("Histograma de residuos")
    fig.suptitle(titulo); plt.tight_layout(); plt.show()
    print(f"Shapiro-Wilk de los residuos: p = {stats.shapiro(modelo.resid).pvalue:.4f}")

---
# Parte 1 (guiada) · Quiebres de inventario y variabilidad de la demanda

Una distribuidora de repuestos registró, para 484 combinaciones producto-mes, el porcentaje de quiebres de inventario, el coeficiente de variación de la demanda semanal y el costo unitario. El área de planeación quiere saber qué explica los quiebres para ajustar su política de reposición.

In [ ]:
inventarios = cargar_csv("IC08_inventarios_quiebres.csv")
display(inventarios.describe().round(2))
display(inventarios.corr().round(3))
sns.pairplot(inventarios, corner=True, plot_kws={"s": 12, "alpha": .5, "color": AZUL}, diag_kws={"color": AZUL})
plt.show()

### 1.1 Correlación y recta de mínimos cuadrados

La prueba sobre la correlación (H₀: ρ = 0) es equivalente a la prueba sobre la pendiente (H₀: β₁ = 0) en la regresión simple.

In [ ]:
r, p = stats.pearsonr(inventarios["variabilidad_demanda_pct"], inventarios["quiebres_inventario_pct"])
print(f"r de Pearson = {r:.3f}   p-valor = {p:.3g}")

m1 = smf.ols("quiebres_inventario_pct ~ variabilidad_demanda_pct", data=inventarios).fit()
print(m1.summary())

In [ ]:
b0, b1 = m1.params
ax = sns.regplot(data=inventarios, x="variabilidad_demanda_pct", y="quiebres_inventario_pct",
                 scatter_kws={"s": 12, "alpha": .5, "color": AZUL}, line_kws={"color": CORAL})
ax.set_title(f"quiebres = {b0:.2f} + {b1:.3f} · variabilidad   (R² = {m1.rsquared:.3f})")
plt.show()

**Pregunta 1.1.** Interprete en el contexto la pendiente, el intercepto y el R². ¿La relación es estadísticamente significativa?

> **✅ Respuesta modelo (docente)**
>
> - **Pendiente (≈ 0,21):** por cada punto porcentual adicional en el coeficiente de variación de la demanda, el porcentaje de quiebres aumenta en promedio unos 0,21 puntos. Su IC 95 % no contiene el cero y p < 0,001: la relación es significativa.
> - **Intercepto (≈ 1,8):** sería el porcentaje medio de quiebres de un producto con demanda perfectamente estable (CV = 0). Es una extrapolación, porque en los datos el CV mínimo es ≈ 9 %; se interpreta con cautela.
> - **R² (≈ 0,40):** la variabilidad de la demanda explica cerca del 40 % de la variación de los quiebres. Es una relación moderada: el 60 % restante depende de otros factores (tiempo de entrega del proveedor, política de inventario de seguridad, etc.).

### 1.2 Intervalo de confianza para la media e intervalo de predicción

Para un producto con CV de demanda de 50 %:
- el **intervalo de confianza** estima el porcentaje **medio** de quiebres de todos los productos con ese CV;
- el **intervalo de predicción** acota el porcentaje de quiebres de **un producto en particular**. Siempre es más ancho.

In [ ]:
nuevo = pd.DataFrame({"variabilidad_demanda_pct": [20, 35, 50]})
pred = m1.get_prediction(nuevo).summary_frame(alpha=0.05)
display(pd.concat([nuevo, pred[["mean", "mean_ci_lower", "mean_ci_upper", "obs_ci_lower", "obs_ci_upper"]]], axis=1).round(2)
        .rename(columns={"mean": "estimación", "mean_ci_lower": "IC media inf", "mean_ci_upper": "IC media sup",
                         "obs_ci_lower": "IP inf", "obs_ci_upper": "IP sup"}))

malla = pd.DataFrame({"variabilidad_demanda_pct": np.linspace(inventarios["variabilidad_demanda_pct"].min(), inventarios["variabilidad_demanda_pct"].max(), 100)})
bandas = m1.get_prediction(malla).summary_frame()
plt.scatter(inventarios["variabilidad_demanda_pct"], inventarios["quiebres_inventario_pct"], s=10, color=AZUL, alpha=.4)
plt.plot(malla, bandas["mean"], color=CORAL)
plt.fill_between(malla.iloc[:, 0], bandas["mean_ci_lower"], bandas["mean_ci_upper"], color=CORAL, alpha=.35, label="IC 95 % de la media")
plt.fill_between(malla.iloc[:, 0], bandas["obs_ci_lower"], bandas["obs_ci_upper"], color=CORAL, alpha=.12, label="IP 95 % de una observación")
plt.xlabel("variabilidad de la demanda (%)"); plt.ylabel("quiebres (%)"); plt.legend(); plt.show()

### 1.3 Validación de supuestos

In [ ]:
graficos_residuos(m1, "Modelo simple: quiebres ~ variabilidad")

**Pregunta 1.2.** ¿Los residuos respaldan los supuestos del modelo? ¿Qué limitación tiene usar el intervalo de predicción para productos con muy baja variabilidad?

> **✅ Respuesta modelo (docente)**
>
> Los residuos no muestran una curva clara frente a los valores ajustados, por lo que la linealidad es razonable. La dispersión es relativamente constante, aunque se aprecia un "piso": el porcentaje de quiebres no puede ser negativo y hay registros en 0 %. Por eso el histograma de residuos puede mostrar algo de asimetría. Para productos con CV bajo, el límite inferior del intervalo de predicción puede ser **negativo**, un valor imposible: el modelo lineal no respeta que la variable está acotada en 0. Eso no invalida el análisis, pero hay que reportar el límite inferior como 0 y reconocer la limitación.

### 1.4 ¿El costo unitario aporta información adicional?

In [ ]:
m2 = smf.ols("quiebres_inventario_pct ~ variabilidad_demanda_pct + costo_unitario_miles", data=inventarios).fit()
print(m2.summary().tables[1])
print(f"R² simple = {m1.rsquared:.4f}   R² múltiple = {m2.rsquared:.4f}   R² ajustado múltiple = {m2.rsquared_adj:.4f}")

**Pregunta 1.3.** ¿Conviene incluir el costo unitario en el modelo? Justifique con la prueba t del coeficiente y con el cambio en el R² ajustado.

> **✅ Respuesta modelo (docente)**
>
> No. El coeficiente del costo unitario no es significativo (p ≈ 0,11) y el R² ajustado prácticamente no cambia. El R² siempre aumenta un poco al agregar variables, por eso se compara el **R² ajustado**, que penaliza el número de predictores. Se prefiere el modelo simple: explica lo mismo y es más fácil de comunicar.

---
# Parte 2 (semi-guiada) · ¿Qué explica el consumo de combustible de la flota?

Una empresa de transporte registró 326 viajes con su consumo (L/100 km), el tipo de vehículo, el perfil de la ruta, el día de la semana, la carga promedio y la experiencia del conductor.

**2.1** Explore la relación del consumo con cada variable: un diagrama de dispersión con la carga (coloreado por tipo de vehículo) y diagramas de caja por tipo de vehículo, perfil de ruta y día.

In [ ]:
flota = cargar_csv("IC10_consumo_combustible_flota.csv")
fig, ax = plt.subplots(2, 2, figsize=(13, 8))
sns.scatterplot(data=flota, x="carga_promedio_ton", y="consumo_combustible_l_100km", hue="tipo_vehiculo", s=18, ax=ax[0, 0])
sns.boxplot(data=flota, x="tipo_vehiculo", y="consumo_combustible_l_100km", ax=ax[0, 1])
sns.boxplot(data=flota, x="perfil_ruta", y="consumo_combustible_l_100km", order=["Plana", "Mixta", "Montañosa"], ax=ax[1, 0])
sns.boxplot(data=flota, x="dia_semana", y="consumo_combustible_l_100km", order=["Lunes", "Martes", "Miercoles", "Jueves", "Viernes", "Sabado"], ax=ax[1, 1])
plt.tight_layout(); plt.show()
print(f"r(carga, consumo) = {flota['carga_promedio_ton'].corr(flota['consumo_combustible_l_100km']):.3f}")

**2.2** Ajuste un modelo múltiple con **todas** las variables. Las categóricas se incluyen con `C(variable)`; statsmodels crea automáticamente las variables indicadoras (dummies) tomando la primera categoría como referencia. Guarde el modelo en `m_completo`.

In [ ]:
m_completo = smf.ols("consumo_combustible_l_100km ~ carga_promedio_ton + experiencia_conductor_anios"
                     " + C(tipo_vehiculo) + C(perfil_ruta, Treatment('Plana')) + C(dia_semana)", data=flota).fit()
print(m_completo.summary())

**2.3** ¿El día de la semana aporta al modelo? Compare el modelo completo con uno sin `dia_semana` mediante la **prueba F parcial** (`anova_lm(modelo_reducido, modelo_completo)`).

In [ ]:
m_sin_dia = smf.ols("consumo_combustible_l_100km ~ carga_promedio_ton + experiencia_conductor_anios"
                    " + C(tipo_vehiculo) + C(perfil_ruta, Treatment('Plana'))", data=flota).fit()
display(sm.stats.anova_lm(m_sin_dia, m_completo).round(4))
print(f"R² ajustado: completo = {m_completo.rsquared_adj:.4f}   sin día = {m_sin_dia.rsquared_adj:.4f}")

**2.4** Revise la **multicolinealidad** con el factor de inflación de varianza (VIF) del modelo sin día y valide los residuos.

In [ ]:
X = m_sin_dia.model.exog
vif = pd.Series([variance_inflation_factor(X, i) for i in range(1, X.shape[1])], index=m_sin_dia.model.exog_names[1:])
display(vif.round(2).to_frame("VIF"))
display(flota.groupby("tipo_vehiculo")["carga_promedio_ton"].describe().round(2))
graficos_residuos(m_sin_dia, "Consumo ~ carga + experiencia + tipo de vehículo + perfil de ruta")

**Pregunta 2.1.** Interprete los coeficientes del modelo final para un gerente de flota. ¿Por qué el efecto de la carga es mucho menor en el modelo múltiple de lo que sugiere su correlación simple con el consumo (r ≈ 0,85)?

> **✅ Respuesta modelo (docente)**
>
> Manteniendo constantes las demás variables:
> - un **camión mediano** consume en promedio ≈ 6,5 L/100 km más que uno liviano y un **pesado** ≈ 15,6 L/100 km más;
> - una **ruta montañosa** aumenta el consumo ≈ 7 L/100 km respecto a una plana, y una mixta ≈ 1,9 L/100 km;
> - cada **tonelada** adicional de carga aumenta el consumo ≈ 0,44 L/100 km;
> - la **experiencia del conductor** y el **día de la semana** no tienen efecto significativo (la prueba F parcial no rechaza para el día).
>
> La correlación simple entre carga y consumo es alta porque los camiones pesados llevan más carga **y** consumen más por su tamaño: la carga "se lleva el crédito" del tipo de vehículo. Al controlar por el tipo de vehículo, queda solo el efecto propio de la carga. El VIF alto de la carga y de las dummies del tipo de vehículo confirma esa colinealidad: los coeficientes siguen siendo válidos, pero son menos precisos y no deben interpretarse aisladamente.

**Pregunta 2.2.** Use el modelo final para estimar el consumo de un camión pesado con 15 toneladas, conductor con 10 años de experiencia, en ruta montañosa. Reporte la estimación y el intervalo de predicción del 95 %.

In [ ]:
caso = pd.DataFrame({"carga_promedio_ton": [15], "experiencia_conductor_anios": [10],
                     "tipo_vehiculo": ["Camion pesado"], "perfil_ruta": ["Montañosa"]})
m_sin_dia.get_prediction(caso).summary_frame(alpha=0.05).round(2)

---
# Parte 3 (autónoma) · Condiciones ergonómicas y frecuencia cardiaca

El equipo de seguridad y salud en el trabajo midió en 731 observaciones la frecuencia cardiaca del trabajador, la temperatura del puesto, el tiempo de postura estática y el nivel de ruido. Sospecha que estas condiciones elevan la frecuencia cardiaca.

Realice un análisis completo: correlaciones con sus pruebas, regresiones simples y un modelo múltiple, y concluya. Sea crítico con lo que encuentre.

In [ ]:
ergo = cargar_csv("IC06_condiciones_ergonomicas.csv")
y = "frecuencia_cardiaca_lpm"
for x in ["temperatura_ambiente_c", "tiempo_postura_estatica_min", "nivel_ruido_dba"]:
    r, p = stats.pearsonr(ergo[x], ergo[y])
    print(f"{x:30s} r = {r:+.3f}   p = {p:.3f}")
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, x in zip(ax, ["temperatura_ambiente_c", "tiempo_postura_estatica_min", "nivel_ruido_dba"]):
    sns.regplot(data=ergo, x=x, y=y, ax=a, scatter_kws={"s": 8, "alpha": .4, "color": AZUL}, line_kws={"color": CORAL})
plt.tight_layout(); plt.show()
m_ergo = smf.ols(f"{y} ~ temperatura_ambiente_c + tiempo_postura_estatica_min + nivel_ruido_dba", data=ergo).fit()
print(m_ergo.summary().tables[0]); print(m_ergo.summary().tables[1])

**Pregunta 3.1.** ¿Qué concluye? ¿Significa esto que las condiciones del puesto no afectan la salud de los trabajadores?

> **✅ Respuesta modelo (docente)**
>
> No hay evidencia de relación lineal: todas las correlaciones son prácticamente nulas (|r| < 0,06, p > 0,1), ninguna pendiente es significativa, la prueba F global no rechaza y el R² es menor que 1 %. El modelo no sirve para predecir la frecuencia cardiaca. Esto **no** demuestra que las condiciones sean inofensivas: (a) puede haber relaciones no lineales o con umbral (por ejemplo, solo por encima de 85 dBA); (b) la frecuencia cardiaca depende mucho de la persona y de la tarea, variables que no se midieron; (c) los efectos pueden ser acumulativos y no aparecer en una medición puntual. La recomendación es rediseñar el estudio (mediciones repetidas por trabajador, registrar la carga física de la tarea) antes de sacar conclusiones, y no "buscar" un resultado forzando el modelo.